# ChurnSignal — Milestone 1: Dataset Reconnaissance
**Learning-First SaaS Customer Churn Prediction**

Welcome to **ChurnSignal**! In this milestone, we conduct pure exploratory data reconnaissance on our customer dataset.

### Core Philosophy:
> *"Never jump directly into model training before understanding the data generating process, label semantics, temporal dynamics, and potential leakage risks."*

### Schema Under Inspection
The dataset tracks monthly customer snapshots with the following columns:
- `user_id`: Unique identifier for each customer account
- `month`: Calendar month timestamp
- `tenure_month`: Customer tenure in months (starting at 0)
- `plan_type`: Subscription tier (`Starter`, `Pro`, `Business`, `Enterprise`)
- `monthly_price`: Monthly subscription price
- `mrr`: Monthly Recurring Revenue
- `sessions`: Number of user web/app sessions during the month
- `feature_usage_score`: Aggregate activity score (0–100)
- `support_tickets`: Number of support inquiries submitted
- `payment_failures`: Number of failed billing attempts
- `churned`: Churn event occurred at the end of the current month
- `churned_next_month`: Churn event will occur in the subsequent month
- `nps_score`: Net Promoter Score rating (0–10)
- `product_incident`: Platform incident indicator (0 or 1)
- `active_seats`: Number of active seats utilized by the account


In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Configure pandas display
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

print('Libraries successfully imported.')

## Step 1: Load the Dataset and Report Shape

### Why this matters:
The dataset is hosted on the Hugging Face Hub under `arti199919/synthetic-saas-churn-sample`. This repository contains two tables:
1. `user_monthly.parquet` (79,817 rows, customer monthly time-series)
2. `users.parquet` (5,000 rows, customer static profiles)

If you invoke `load_dataset('arti199919/synthetic-saas-churn-sample')` without arguments, Hugging Face tries to merge both parquet files into one table, which fails with a `CastError` due to disparate schemas. We must specify `data_files='user_monthly.parquet'`.


In [ ]:
# Step 1: Load dataset
local_path = Path('data/user_monthly.parquet')

try:
    from datasets import load_dataset
    if local_path.exists():
        ds = load_dataset('parquet', data_files=str(local_path))
    else:
        ds = load_dataset('arti199919/synthetic-saas-churn-sample', data_files='user_monthly.parquet')
    df = ds['train'].to_pandas()
    print('Loaded via Hugging Face `load_dataset`.')
except Exception as e:
    print(f'Fallback to direct pandas load ({e})...')
    df = pd.read_parquet(local_path)

print(f'Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')

## Step 2: Data Types and Sample Rows

### Why this matters:
Confirming data types ensures categorical columns are recognized, timestamps are properly parsed, and numeric ranges match expectations.


In [ ]:
# Step 2: Data types and preview
print('--- Column Data Types & Unique Counts ---')
dtype_table = pd.DataFrame({
    'Data Type': df.dtypes.astype(str),
    'Non-Null Count': df.notnull().sum(),
    'Unique Values': df.nunique()
})
display(dtype_table)

print('\n--- First 3 Sample Records ---')
display(df.head(3))

## Step 3: Missing Values and Duplicate Rows

### Why this matters:
Missing data requires imputation strategies, and unexpected duplicate records can bias model parameters or corrupt cross-validation.


In [ ]:
# Step 3: Missing values and duplicates
missing_counts = df.isnull().sum()
print('Missing values per column:')
print(missing_counts[missing_counts > 0] if missing_counts.sum() > 0 else 'Zero missing values across all columns.')

exact_dupes = df.duplicated().sum()
key_dupes = df.duplicated(subset=['user_id', 'month']).sum()

print(f'\nExact duplicate rows: {exact_dupes}')
print(f'Duplicate (user_id, month) pairs: {key_dupes}')

## Step 4: Count Unique Users and Unique Months

### Why this matters:
Establishes the scale of the customer base and the temporal span of our historical observation window.


In [ ]:
# Step 4: Unique entities
n_users = df['user_id'].nunique()
n_months = df['month'].nunique()
min_date = df['month'].min().strftime('%Y-%m-%d')
max_date = df['month'].max().strftime('%Y-%m-%d')

print(f'Unique Customers (user_id) : {n_users:,}')
print(f'Unique Calendar Months     : {n_months}')
print(f'Observation Window         : {min_date} to {max_date}')
print('\nChronological Timeline:')
print(df['month'].dt.strftime('%Y-%m').unique().tolist())

## Step 5: User Record Multiplicity (Longitudinal Structure)

### Why this matters:
If users appear multiple times across months, observations are **not independent and identically distributed (i.i.d.)**. Treating them as independent leads to data leakage if records from the same user are scattered randomly across train and test sets.


In [ ]:
# Step 5: User history depth
records_per_user = df.groupby('user_id').size()
print('Summary statistics for records per customer:')
print(records_per_user.describe())

single_row_users = (records_per_user == 1).sum()
multi_row_users = (records_per_user > 1).sum()
print(f'\nCustomers with only 1 month  : {single_row_users:,} ({single_row_users/len(records_per_user)*100:.2f}%)')
print(f'Customers with > 1 month     : {multi_row_users:,} ({multi_row_users/len(records_per_user)*100:.2f}%)')

## Step 6: Distribution of `churned` and `churned_next_month`

### Why this matters:
Class imbalance dictates our evaluation strategy. In high-imbalance problems, classification accuracy is deceptive; we must focus on Precision, Recall, PR-AUC, and ROC-AUC.


In [ ]:
# Step 6: Target distributions
churn_curr = pd.DataFrame({
    'Count': df['churned'].value_counts(),
    'Percentage (%)': df['churned'].value_counts(normalize=True) * 100
})
print('Distribution of `churned` (current month):')
display(churn_curr)

churn_next = pd.DataFrame({
    'Count': df['churned_next_month'].value_counts(),
    'Percentage (%)': df['churned_next_month'].value_counts(normalize=True) * 100
})
print('\nDistribution of `churned_next_month` (next month target):')
display(churn_next)

## Step 7: Churn Rates by Month (Temporal Dynamics)

### Why this matters:
SaaS business metrics evolve over time. Examining churn rates by month helps uncover macroeconomic seasonality, product release impacts, or data-logging anomalies.


In [ ]:
# Step 7: Monthly churn trends
monthly_summary = df.groupby(df['month'].dt.strftime('%Y-%m')).agg(
    active_customers=('user_id', 'count'),
    current_churn_count=('churned', 'sum'),
    current_churn_rate_pct=('churned', lambda x: x.mean() * 100),
    next_month_churn_count=('churned_next_month', 'sum'),
    next_month_churn_rate_pct=('churned_next_month', lambda x: x.mean() * 100)
)
display(monthly_summary)

print('\nAlert: Look closely at 2025-12 in the table above!')
print('In 2025-12, `churned_next_month` is 0 across all 3,931 accounts because 2026-01 was not recorded.')

## Step 8: Relationship Between `churned` and `churned_next_month`

### Why this matters:
Understanding the precise lag/lead relationship between these two labels prevents fatal target leakage.


In [ ]:
# Step 8: Cross-tabulation and state transitions
print('--- Contemporaneous Crosstab (Month T) ---')
display(pd.crosstab(df['churned'], df['churned_next_month'], 
                    rownames=['churned (Month T)'], 
                    colnames=['churned_next_month (Month T)'], 
                    margins=True))

# Check user trajectories: what happens after churned == 1?
df_sorted = df.sort_values(['user_id', 'month']).copy()
df_sorted['prev_churned'] = df_sorted.groupby('user_id')['churned'].shift(1)
post_churn_count = (df_sorted['prev_churned'] == 1).sum()

print(f'\nObservations recorded after customer churned: {post_churn_count}')
print('Conclusion: Churn is an absorbing state. Customers exit the dataset immediately upon churning.')

## Step 9: Check Whether `active_seat` is Binary or a Count

### Why this matters:
Clarifies whether the feature represents an account flag (active/inactive) or seat volume (team size), which determines whether it should be treated as numeric or boolean.


In [ ]:
# Step 9: active_seats inspection
col_name = 'active_seats' if 'active_seats' in df.columns else 'active_seat'
print(f'Inspecting column: `{col_name}`')
print(f'Data Type: {df[col_name].dtype}')
print(f'Is Binary (only 0 and 1)? {df[col_name].isin([0, 1]).all()}')
print(f'Distinct Values Count: {df[col_name].nunique()}')
print(f'Range: [{df[col_name].min()}, {df[col_name].max()}]')
print(f'Mean: {df[col_name].mean():.2f}, Median: {df[col_name].median():.1f}')

print('\nTop 10 most common seat counts:')
print(df[col_name].value_counts().head(10))

## Step 10: Numeric Feature Correlations

### Why this matters:
Identifies multicollinearity (which destabilizes linear models) and surfaces leading indicators (features correlated with future churn).


In [ ]:
# Step 10: Correlation analysis
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[numeric_cols].corr()

print('--- Full Correlation Matrix ---')
display(corr_matrix.round(3))

print('\n--- Correlation with `churned_next_month` ---')
display(corr_matrix[['churned_next_month']].sort_values(by='churned_next_month', ascending=False).round(4))

print('\nKey Finding: Check identity between monthly_price and mrr:')
print('Is monthly_price == mrr everywhere?', (df['monthly_price'] == df['mrr']).all())

## Step 11: Identify Target Leakage Risks & Ambiguities

### Target Leakage Risks:
1. **Contemporaneous `churned` Feature**: Using `churned` to predict `churned_next_month` causes leakage because when `churned == 1`, `churned_next_month` is forced to 0. In production, we only score active customers (`churned == 0`).
2. **Right-Censored Dec 2025 Target**: In 2025-12, all 3,931 accounts have `churned_next_month = 0` simply because January 2026 was not observed. Training or testing on December 2025 injects false negative labels.
3. **Exact Redundancy**: `monthly_price` and `mrr` are identical down to the cent. Retaining both causes perfect multicollinearity.

### Ambiguities Needing Business Clarification:
- **`active_seats`**: Is this peak concurrent seats, active user logins during the month, or purchased license volume?
- **`nps_score`**: In SaaS companies, NPS is rarely answered monthly by every user. Is this value forward-filled from past surveys?
- **`product_incident`**: Does this signify an infrastructure-wide outage or a customer-specific bug?
- **`feature_usage_score`**: How is this 0–100 index normalized and aggregated?


## Step 12: Chronological Train/Test Split Feasibility

### Why this matters:
Random K-Fold cross-validation destroys the temporal sequence of panel data and causes **identity leakage** (the same customer appearing in both train and test sets). A **chronological split** is essential.


In [ ]:
# Step 12: Chronological Split Demonstration
train_mask = (df['month'] >= '2024-01-01') & (df['month'] <= '2025-06-01')
val_mask   = (df['month'] >= '2025-07-01') & (df['month'] <= '2025-09-01')
test_mask  = (df['month'] >= '2025-10-01') & (df['month'] <= '2025-11-01')
excl_mask  = (df['month'] == '2025-12-01')

print(f'Train Set      (2024-01 to 2025-06) : {train_mask.sum():>5,} rows | Churn Next Month: {df.loc[train_mask, "churned_next_month"].sum():>3} ({df.loc[train_mask, "churned_next_month"].mean()*100:.2f}%)')
print(f'Validation Set (2025-07 to 2025-09) : {val_mask.sum():>5,} rows | Churn Next Month: {df.loc[val_mask, "churned_next_month"].sum():>3} ({df.loc[val_mask, "churned_next_month"].mean()*100:.2f}%)')
print(f'Test Set       (2025-10 to 2025-11) : {test_mask.sum():>5,} rows | Churn Next Month: {df.loc[test_mask, "churned_next_month"].sum():>3} ({df.loc[test_mask, "churned_next_month"].mean()*100:.2f}%)')
print(f'Excluded       (2025-12)            : {excl_mask.sum():>5,} rows | (Truncated target artifact)')

# Verify complete partitioning
total_assigned = train_mask.sum() + val_mask.sum() + test_mask.sum() + excl_mask.sum()
print(f'\nTotal rows accounted for: {total_assigned:,} / {len(df):,}')

## Milestone 1 Reconnaissance Summary

1. **Dataset Structure**: 79,817 longitudinal monthly records across 5,000 customers over 24 calendar months (Jan 2024 to Dec 2025).
2. **Data Hygiene**: Zero missing values, zero duplicate customer-month pairs.
3. **ML Target**: The true production target is `churned_next_month` (lead label for month $T+1$), exhibiting ~1.32% positive class imbalance.
4. **Key Finding**: `churned == 1` is an absorbing terminal state. December 2025 has truncated zero labels for `churned_next_month`.
5. **Collinearity**: `monthly_price` and `mrr` are 100% identical.
6. **Next Steps**: Address feature ambiguities, define chronological train/val/test splits, select PR-AUC/ROC-AUC evaluation metrics, and engineer lag/rolling features before training baseline models.
